# Phase C: Planning failure analysis

**Question:** when TF++ crashed, was the *plan* already unsafe, even though the actor was visible?

TF++'s plan at each step (20 per second) has two parts:

- a **path**: 10 points ahead of the car (`predicted_path`);
- a **target speed** (`target_speed_scalar`), derived from 8 speed-class probabilities.

The controller (Phase D) then turns the plan into steering, throttle and brake.

| Section | Question |
|---|---|
| 1 | What did the target speed do before each crash? |
| 2 | **Retrospective check:** if the car had followed the plan exactly, would it have hit the actor's *real* future path? |
| 3 | **Intent-aware check:** could TF++ have seen the conflict from its *own* boxes? |
| 4 | Is the planner more *unstable* before crashes than during successful manoeuvres? |
| 5 | Did the plan follow the route? |
| 6 | One summary line per collision |

**Data:** the 220 Bench2Drive routes with schema-v2 logs. Ground truth is used only for diagnosis, never as an input a monitor could use. The main population is the three core scenario families (junction turns, merges, obstacle detours); other collisions are shown separately.

In [ ]:
import sys, csv, json, math, collections
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
import ew_analysis as ew
import pce_analysis as pce

RESULTS = REPO / "results" / "bench2drive_v2"
OUT = RESULTS / "pce"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(rows, cols=None, fmt=None):
    cols = cols or list(rows[0]); fmt = fmt or {}
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(fmt[c](r[c]) if c in fmt and isinstance(r.get(c), (int, float)) and r.get(c) == r.get(c)
                                     else (f"{r[c]:.2f}" if isinstance(r.get(c), float) else str(r.get(c, ""))) for c in cols) + " |")
    display(Markdown("\n".join(out)))

def save_csv(name, rows):
    with open(OUT / f"{name}.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

pct = lambda v: f"{100 * v:.0f}%"

# shared data: collision events and features from the early-warning cache, plans/poses from the pce cache
routes, ckpts, EW = ew.build_cache(RESULTS, REPO / "carla_garage/leaderboard/data/bench2drive220.xml", RESULTS / "early_warning" / "cache.pkl")
PC = pce.build_cache(RESULTS, OUT / "cache.pkl")
events = ew.event_table(routes, ckpts, EW)
CORE = ("junction", "lane_merge", "obstacle_detour")
crashes = [e for e in events if e["counted"] and e["clean_history"] and e["step"] >= 60]
for e in crashes:
    e["core"] = e["family"] in CORE
    e["ego_speed"] = e["v_pre"]          # ego speed 0.1 s before contact
print(f"collisions analysed: {len(crashes)} (counted by the leaderboard, no other contact in the 3 s before, >= 3 s of log)")
print("  by family:", dict(collections.Counter(e["family"] for e in crashes)))

## 1. Target speed before each crash

The grey band is successful junction turns, aligned at the turn's peak, for comparison. A planner that recognised the danger would cut its target speed well before impact.

In [ ]:
anchors = [a for a in ew.maneuver_anchors(routes, EW, events) if a["label"] == "junction turn"]
tt = np.arange(-60, 1) * ew.DT
def trace(rid, end, key, src="pc"):
    x = PC[rid][key] if src == "pc" else ew.step_features(EW[rid])[key]
    return x[end - 60:end + 1] if end >= 60 else None

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, key, lab in ((axes[0], "tss", "TF++ target speed (m/s)"), (axes[1], "speed", "actual speed (m/s)")):
    A = np.array([trace(e["route"], e["step"], key) for e in crashes if e["core"]])
    S = np.array([x for a in anchors if (x := trace(a["route"], a["step"], key)) is not None])
    for M_, c, l in ((A, "#c0392b", f"core-family crashes (n={len(A)})"), (S, "#7f8c8d", f"successful junction turns (n={len(S)})")):
        m, se = M_.mean(0), M_.std(0) / np.sqrt(len(M_))
        ax.plot(tt, m, color=c, label=l); ax.fill_between(tt, m - 1.96 * se, m + 1.96 * se, color=c, alpha=0.15)
    ax.set_xlabel("seconds before impact / turn peak"); ax.set_title(lab)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "C1_target_speed.png", bbox_inches="tight"); plt.show()

rows = []
for e in crashes:
    d = PC[e["route"]]; s = e["step"]
    e["tss_3s"], e["tss_1s"], e["tss_0"] = d["tss"][s - 60], d["tss"][s - 20], d["tss"][s]
    e["speed_1s"] = d["speed"][s - 20]
    e["tss_cut_before_1s"] = bool(d["tss"][s - 20] < d["tss"][s - 60] - 2.0)
for grp, sel in (("core families", [e for e in crashes if e["core"]]), ("other families", [e for e in crashes if not e["core"]])):
    rows.append({"group": grp, "crashes": len(sel),
                 "median target speed 3 s before": float(np.median([e["tss_3s"] for e in sel])),
                 "median target speed 1 s before": float(np.median([e["tss_1s"] for e in sel])),
                 "target speed cut by >2 m/s before the last second": float(np.mean([e["tss_cut_before_1s"] for e in sel]))})
table(rows, fmt={"target speed cut by >2 m/s before the last second": pct})

## 2. Retrospective check: did the plan lead into the actor's real future path?

For each collision, take the plan TF++ made at 3, 2, 1.5, 1 and 0.5 s before impact and roll it forward as intended:

- **Ego:** along the planned path, at a speed that moves from the current speed toward the target speed (+2 / −4 m/s²).
- **Struck actor:** where it **actually** went, from ground truth.

The plan is called **in conflict** if the car would come within 0.5 m of the actor, measured between the boxes, within the next 3 s. The **time gap** is how far apart in time the two would reach the same spot; it shows whether the paths cross without the timing matching.

A plan that conflicts with the actor's real motion is direct evidence of a planning error. Section 3 asks whether TF++ could have known.

In [ ]:
H = (3.0, 2.0, 1.5, 1.0, 0.5)
for e in crashes:
    ev = PC[e["route"]]["events"][e["event"]]
    e["retro"] = {h: pce.retro_check(PC[e["route"]], ev, e["step"] - int(h * ew.HZ)) for h in H}
rows = []
for h in H:
    for grp, sel in (("core", [e for e in crashes if e["core"]]), ("all", crashes)):
        v = [e["retro"][h] for e in sel if not np.isnan(e["retro"][h][1])]
        rows.append({"seconds before impact": h, "crashes": grp, "actor logged": len(v),
                     "plan in conflict with actor's real path": float(np.mean([x[0] < np.inf for x in v])),
                     "paths cross at all (any timing)": float(np.mean([x[2] < np.inf for x in v])),
                     "median closest approach (m)": float(np.median([x[1] for x in v]))})
table(rows, fmt={"plan in conflict with actor's real path": pct, "paths cross at all (any timing)": pct})
save_csv("C2_retrospective_conflict", rows)
print("Sanity check: 0.5 s before impact nearly every plan should already be in conflict.")

## 3. Intent-aware check: could TF++ have seen it from its own boxes?

This repeats the check above, but with information TF++ actually had at that moment:

- the actor's position from **TF++'s own predicted box**;
- the actor's velocity from frame-to-frame **tracking** of that box (TF++'s box speed output is always 0);
- constant velocity into the future.

The **CTRV** row ignores TF++'s plan altogether: it extends the ego's current speed and turn rate. The table shows how early each method first flags the conflict.

In [ ]:
def onset(tr, key, thr=3.0):
    ks = sorted(k for k, r in tr.items() if r.get(key, np.inf) < thr)
    if not ks or ks[-1] < -5:
        return 0.0
    st = ks[-1]
    for k in ks[::-1][1:]:
        if st - k <= 5: st = k
        else: break
    return -st * ew.DT

dyn = [e for e in crashes if not e["other_type"].startswith("static")]
for e in dyn:
    tr = {r["k"]: r for r in EW[e["route"]]["events"][e["event"]]["track"]}
    e["onset_boxes_plan"] = onset(tr, "pred_ttc_plan")
    e["onset_gt_plan"] = onset(tr, "gt_ttc_plan")
    e["onset_gt_ctrv"] = onset(tr, "gt_ttc_ctrv")
    e["onset_retro"] = max([h for h in H if e["retro"][h][0] < np.inf], default=0.0)
rows = []
for key, lab in (("onset_retro", "plan vs actor's REAL future (hindsight)"),
                 ("onset_gt_plan", "plan vs actor at constant velocity (GT position)"),
                 ("onset_boxes_plan", "plan vs TF++'s own tracked boxes (what TF++ could know)"),
                 ("onset_gt_ctrv", "CTRV ego (ignores the plan) vs GT actor")):
    v = np.array([e[key] for e in dyn if e["core"]])
    rows.append({"conflict detected by": lab, "flagged >= 1 s before impact": float((v >= 1).mean()),
                 ">= 2 s": float((v >= 2).mean()), "median first flag (s before impact)": float(np.median(v))})
display(Markdown(f"Core-family collisions with a vehicle or pedestrian: **{sum(e['core'] for e in dyn)}**"))
table(rows, fmt={"flagged >= 1 s before impact": pct, ">= 2 s": pct})
save_csv("C3_conflict_onset", rows)

## 4. Is the planner unstable before crashes?

The proposal's temporal signals are:

- path disagreement between consecutive plans (after correcting for the car's own motion);
- path end-point jitter;
- the change in the target-speed distribution (JS divergence);
- entropy and top-class margin of that distribution.

The comparison:

- **Crash windows:** 3 s to 1 s before impact. This is the part that could still be acted on.
- **Safe windows:** the 2 s before the peak of a *successful* manoeuvre of the same kind.

In the AUROC table, above 0.5 means the signal is higher before crashes, below 0.5 means it is higher in the successful manoeuvre, and about 0.5 means no difference.

In [ ]:
from sklearn.metrics import roc_auc_score
safe = collections.defaultdict(list)
for a in ew.maneuver_anchors(routes, EW, events):
    safe[a["label"]].append(a)
F = {}
def wmean(rid, end, key):
    if rid not in F:
        F[rid] = ew.step_features(EW[rid])
    return F[rid][key][end - 40:end].mean()
SIG = [("path_disagree_mean_1s", "path disagreement"), ("path_end_lat_std_1s", "path end-point jitter"),
       ("path_curvature", "path curvature"), ("speed_js_mean_1s", "target-speed change (JS)"),
       ("speed_entropy", "speed entropy"), ("speed_margin", "speed top-class margin"), ("target_speed", "target speed")]
pairs = [("junction crashes vs successful junction turns", [e for e in crashes if e["situation"] == "junction turn / crossing"], safe["junction turn"]),
         ("bypass crashes vs successful bypasses", [e for e in crashes if e["situation"] == "obstacle bypass"], safe["obstacle bypass"]),
         ("all core crashes vs normal driving", [e for e in crashes if e["core"]], safe["normal driving"])]
rows = []
for key, lab in SIG:
    r = {"signal": lab}
    for name, C_, S_ in pairs:
        a = [wmean(e["route"], e["step"] - 20, key) for e in C_]
        b = [wmean(x["route"], x["step"], key) for x in S_]
        r[name] = roc_auc_score(np.r_[np.ones(len(a)), np.zeros(len(b))], np.r_[a, b])
    rows.append(r)
display(Markdown("AUROC (n = " + ", ".join(f"{len(C_)} vs {len(S_)}" for _, C_, S_ in pairs) + ")"))
table(rows)
save_csv("C4_instability_vs_safe", rows)

## 5. Did the plan follow the route?

**Route alignment** is the angle between the direction to TF++'s route target point and the heading at the end of its planned path. It is about 0° when the plan heads where the route goes. A large angle just before a crash would mean TF++ was doing something other than what the route asked.

In [ ]:
rows = []
for name, sel, end_off in (("core crashes (3 s to 1 s before)", [e for e in crashes if e["core"]], -20),
                           ("successful junction turns", safe["junction turn"], 0),
                           ("successful bypasses", safe["obstacle bypass"], 0),
                           ("normal driving", safe["normal driving"], 0)):
    v = np.degrees([wmean(x["route"], x["step"] + end_off, "route_align") for x in sel])
    rows.append({"windows": name, "n": len(v), "median route alignment (deg)": float(np.median(v)),
                 "share > 20 deg off the route": float(np.mean(v > 20))})
table(rows, fmt={"share > 20 deg off the route": pct})

## 6. One line per collision

This is the planning evidence used by Phase E (`results/bench2drive_v2/pce/planning_per_collision.csv`).

In [ ]:
rows = []
for e in crashes:
    tr = {r["k"]: r for r in EW[e["route"]]["events"][e["event"]]["track"]}
    seen = [r for k, r in tr.items() if -30 <= k <= -10 and r.get("in_range")]
    rows.append({"route": e["route"], "scenario": e["scenario"], "family": e["family"], "situation": e["situation"],
                 "other": e["other_type"], "ego speed at impact": round(e["ego_speed"], 1),
                 "detected share (-1.5..-0.5 s)": round(float(np.mean([r["detected"] for r in seen])), 2) if seen else "",
                 "plan conflict (real future) first flag (s)": e.get("onset_retro", max([h for h in H if e["retro"][h][0] < np.inf], default=0.0)),
                 "conflict from TF++ boxes first flag (s)": e.get("onset_boxes_plan", ""),
                 "target speed 3 s / 1 s / 0 s": f"{e['tss_3s']:.1f} / {e['tss_1s']:.1f} / {e['tss_0']:.1f}",
                 "target speed cut before last 1 s": e["tss_cut_before_1s"]})
save_csv("planning_per_collision", rows)
table(sorted(rows, key=lambda r: (r["family"], r["route"])))

## Takeaways

Numbers are for the 53 core-family collisions (junctions, merges, detours) unless stated otherwise.

1. **TF++ speeds up into the crash.** Its median target speed *rises* from 8.1 m/s three seconds before impact to 10.1 m/s one second before. Only 17% of core crashes show a target-speed cut of more than 2 m/s before the last second.
2. **The plan visibly runs into the actor, but mostly late.**

   | Seconds before impact | 3 | 2 | 1.5 | 1 | 0.5 |
   |---|---|---|---|---|---|
   | Plan in conflict with the actor's real future path | 14% | 18% | 18% | 40% | 92% |

   The 92% at 0.5 s confirms the method works. Earlier detection is limited because **TF++'s planned path only reaches about 11 m ahead** (about 1 s at 10 m/s), so a plan made 2–3 s before impact usually does not yet reach the collision point.
3. **TF++ could often have known from its own boxes.** Following its own plan against its own tracked boxes flags the conflict ≥1 s before impact in **52%** of core vehicle and pedestrian collisions (≥2 s in 13%; median 1.0 s). Ignoring the plan (CTRV) flags only 22%. In about half the crashes, the conflict was visible from TF++'s own outputs at least 1 s ahead, and it did not slow down.
4. **Crash plans are stable, not erratic.** Compared with *successful* junction turns, junction crashes have **less** path disagreement (AUROC 0.28), end-point jitter (0.04) and curvature (0.09). Against normal driving the crashes do stand out (curvature 0.78, top-class margin 0.72), but only because they happen during manoeuvres. In bypasses, only the target speed separates crashes from successes (0.81).
5. **The plans follow the route.** Crash plans point a median of 5° from the route target, and only 4% are more than 20° off. TF++ was not going the wrong way; it misjudged *when* it was safe to go.

**Planning failure signature:** a stable, confident plan along the correct route, accelerating into a gap that its own boxes show is closing. This is a gap-acceptance and timing failure, not planner instability.
